# 01 - 原始数据探索（EDA）

在**不运行预处理**的前提下，快速查看 `configs/data.yaml` 指向的**主要原始表**（mcelog、tickets、inventory）及**索引表** `index_features_path`（训练样本行定义；默认演示表由 `bash scripts/build_index.sh smoke` 从 raw 生成）。

**输出**：`outputs/tables/eda_raw_inventory_columns.csv`、`eda_raw_tickets_columns.csv`、`eda_mcelog_summary.csv`。其中 mcelog 为**全文件**汇总：`error_type` 计数按 `data.yaml` 中的 **`mcelog_chunk_rows`** 分块流式读取并累加。

**配置提示**：默认 **`label_horizon_seconds=300`（未来 5 分钟是否故障）**；horizon 对比实验需先 `build_index.sh` / `build_horizon_indexes.sh` 生成匹配索引。

**时间轴**：raw CSV 中 mcelog/tickets 可能为「公元 0001」等低年份；本仓库 `build_index` / preprocess 会将**全局最早 raw 时刻**平移为 `t=0`（相对秒）。索引 CSV 的 `predict_time` 以 **1970-01-01 + 相对秒** 的 ISO 字符串展示；`meta.csv` 中 `predict_ts` 为 **≥ 0 的相对秒**（见 `docs/data_processing_guide.md` §4.1）。

**与 04 流水线 Notebook 的关系**：本 EDA **不跑训练**；冒烟/全量端到端请用 **`04_run_pipelines.ipynb`**（先 EDA 确认 raw 可读，再跑 04 冒烟或全量单元）。

**大数据**：全表扫描可能**耗时长**；若内存紧张可调低 `mcelog_chunk_rows`。


In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
_src = str(_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from dram2cnn.utils.notebook_bootstrap import infer_project_root

PROJECT_ROOT = infer_project_root(cwd=_cwd)
OUT_T = PROJECT_ROOT / "outputs" / "tables"
OUT_T.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /root/workspace


In [2]:
import yaml

with (PROJECT_ROOT / "configs" / "data.yaml").open(encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

paths = {
    "mcelog": Path(data_cfg["mcelog_path"]),
    "tickets": Path(data_cfg["tickets_path"]),
}
inv_raw = data_cfg.get("inventory_path")
if inv_raw:
    paths["inventory"] = Path(inv_raw)
idx_raw = data_cfg.get("index_features_path")
if idx_raw:
    paths["index_features"] = Path(idx_raw)

if not paths["mcelog"].is_absolute():
    paths["mcelog"] = PROJECT_ROOT / paths["mcelog"]
if not paths["tickets"].is_absolute():
    paths["tickets"] = PROJECT_ROOT / paths["tickets"]
if "inventory" in paths and not paths["inventory"].is_absolute():
    paths["inventory"] = PROJECT_ROOT / paths["inventory"]
if "index_features" in paths and not paths["index_features"].is_absolute():
    paths["index_features"] = PROJECT_ROOT / paths["index_features"]

for k, v in paths.items():
    print(k, v, "OK" if v.is_file() else "MISSING")


mcelog /root/workspace/data/raw/mcelog.csv OK
tickets /root/workspace/data/raw/trouble_tickets.csv OK
inventory /root/workspace/data/raw/inventory.csv OK
index_features /root/workspace/data/index_stubs/features_5m_index_demo.csv OK


In [3]:
import pandas as pd

# inventory（data.yaml 中 inventory_path 可为 null）
inv_p = paths.get("inventory")
if inv_p is not None and inv_p.is_file():
    inv = pd.read_csv(inv_p)
    inv_summary = pd.DataFrame({
        "column": inv.columns,
        "dtype": [str(inv[c].dtype) for c in inv.columns],
        "non_null_frac": [inv[c].notna().mean() for c in inv.columns],
        "nunique": [inv[c].nunique(dropna=False) for c in inv.columns],
    })
    inv_summary.to_csv(OUT_T / "eda_raw_inventory_columns.csv", index=False)
    print("inventory rows=", len(inv), "cols=", len(inv.columns))
    print(inv.head(2))
else:
    print("跳过 inventory：未配置 inventory_path 或文件不存在")


inventory rows= 30502 cols= 4
        sid server_manufacturer DRAM_model  DIMM_number
0  Server_1                  M2         C1           24
1  Server_2                  M2         C1           24


In [4]:
import pandas as pd

tk_p = paths["tickets"]
if tk_p.is_file():
    tk = pd.read_csv(tk_p)
    tk_summary = pd.DataFrame({
        "column": tk.columns,
        "dtype": [str(tk[c].dtype) for c in tk.columns],
        "non_null_frac": [tk[c].notna().mean() for c in tk.columns],
    })
    tk_summary.to_csv(OUT_T / "eda_raw_tickets_columns.csv", index=False)
    print("tickets rows=", len(tk), "cols=", len(tk.columns))
    if "sid" in tk.columns:
        print("unique sid (tickets)", tk["sid"].nunique())
    print(tk.head(2))
else:
    print("跳过 tickets：文件不存在")


tickets rows= 3017 cols= 3
unique sid (tickets) 3017
            sid  failure_type          failed_time
0  Server_10003             2  0001-07-17 11:11:07
1  Server_10034             3  0001-08-11 07:35:07


In [5]:
import pandas as pd

# 索引表：训练样本锚点 (sid, predict_time)；冒烟默认 ~500 行
# predict_time 为 1970-01-01 + 相对秒 的 ISO 展示（build_index 产物）；preprocess 后 meta.csv 含 predict_ts（≥0 相对秒）
idx_p = paths.get("index_features")
if idx_p is not None and idx_p.is_file():
    idx_df = pd.read_csv(idx_p, nrows=5)
    with idx_p.open("rb") as f:
        n_idx = sum(1 for _ in f) - 1
    print(f"index_features: {idx_p.name} 约 {n_idx} 行（含表头减 1）")
    print("columns:", list(idx_df.columns))
    print(idx_df.head(3))
    if "index_stubs" in str(idx_p) or "features_5m_index_demo" in str(idx_p):
        print("提示：当前为演示/stub 索引；全量请 build_index.sh full 并用 exp_full_f1_ddp.yaml 或 exp_full_recall_ddp.yaml")
else:
    print("跳过 index_features：未配置或文件不存在 → 可先 bash scripts/build_index.sh smoke")


index_features: features_5m_index_demo.csv 约 500 行（含表头减 1）
columns: ['sid', 'predict_time']
           sid         predict_time
0   Server_720  1970-03-17 02:49:00
1   Server_379  1970-03-31 21:15:17
2  Server_1488  1970-03-17 03:43:41
提示：当前为演示/stub 索引；全量请 build_index.sh full 并用 exp_full_f1_ddp.yaml 或 exp_full_recall_ddp.yaml


In [6]:
import pandas as pd

chunksize = int(data_cfg.get("mcelog_chunk_rows", 1_000_000))
mp = paths["mcelog"]
if not mp.is_file():
    raise FileNotFoundError(f"mcelog 不存在: {mp}")

total_rows = 0
et_counts = {}
n_chunks = 0
for chunk in pd.read_csv(mp, chunksize=chunksize):
    total_rows += len(chunk)
    if "error_type" in chunk.columns:
        vc = chunk["error_type"].value_counts(dropna=False)
        for k, v in vc.items():
            et_counts[str(k)] = et_counts.get(str(k), 0) + int(v)
    n_chunks += 1

out_path = OUT_T / "eda_mcelog_summary.csv"
rows = [{"metric": "chunks_read", "value": n_chunks}, {"metric": "rows_total", "value": total_rows}]
rows += [{"metric": f"error_type_{k}", "value": v} for k, v in sorted(et_counts.items())]
pd.DataFrame(rows).to_csv(out_path, index=False)
print(f"mcelog: 读取 chunk 数={n_chunks}, 总行数={total_rows}")
print("error_type 计数(全文件):", et_counts)
print("已写入:", out_path)


mcelog: 读取 chunk 数=76, 总行数=75214692
error_type 计数(全文件): {'2': 36623454, '1': 38504052, '3': 87186}
已写入: /root/workspace/outputs/tables/eda_mcelog_summary.csv
